# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danielajetunmobi/flyrank/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup — the same cohort, gate and model as ML-08

Rebuilt here rather than imported, so this notebook stands alone and its numbers can be checked
against ML-08's without trusting a shared file. The printed totals must match ML-08 exactly; if they
do not, one of the two notebooks has drifted and nothing below is comparable.

In [1]:
%pip install -q duckdb huggingface_hub pandas numpy scipy scikit-learn matplotlib python-dotenv

import os
import sys
import duckdb
import numpy as np
import pandas as pd
import sklearn
from huggingface_hub import hf_hub_download

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

SEED = 8
print(f"python {sys.version.split()[0]} | pandas {pd.__version__} | "
      f"numpy {np.__version__} | scikit-learn {sklearn.__version__}")

token = os.environ.get("HF_TOKEN")
if not token:
    import getpass
    token = getpass.getpass("Hugging Face token (read): ")

REPO = "FlyRank/internship-warehouse"
MONTHS = ["2025-12", "2026-01", "2026-02", "2026-03", "2026-04", "2026-05", "2026-06"]
daily_files = [hf_hub_download(repo_id=REPO, repo_type="dataset",
               filename=f"fact_content_daily_performance/month={m}/data_0.parquet", token=token)
               for m in MONTHS]
dim_file = hf_hub_download(repo_id=REPO, repo_type="dataset",
                           filename="dim_content.parquet", token=token)
con = duckdb.connect()
REL = "read_parquet([" + ", ".join(f"'{f}'" for f in daily_files) + "])"
D1 = "2026-03-31"

q = f"""
WITH prior AS (
  SELECT content_hash_id, ANY_VALUE(client_hash_id) AS client_hash_id,
    SUM(gsc_impressions) AS impr_90d,
    SUM(gsc_clicks) AS clicks_90d,
    COUNT(*) FILTER (WHERE gsc_impressions > 0) AS days_with_impressions,
    SUM(gsc_sum_position) AS sum_position_90d,
    SUM(ga4_sessions) AS ga4_sessions_90d,
    SUM(ga4_engaged_sessions) AS ga4_engaged_90d,
    SUM(ga4_pageviews) AS ga4_pageviews_90d,
    SUM(scroll_events) AS scroll_events_90d,
    BOOL_OR(ga4_data_available) AS ga4_available,
    SUM(gsc_impressions) FILTER (
        WHERE report_date < DATE '{D1}' - INTERVAL 30 DAY) AS older60_impr,
    SUM(gsc_impressions) FILTER (
        WHERE report_date >= DATE '{D1}' - INTERVAL 30 DAY) AS recent30_impr,
    SUM(gsc_impressions) FILTER (
        WHERE report_date >= DATE '{D1}' - INTERVAL 60 DAY
          AND report_date <  DATE '{D1}' - INTERVAL 30 DAY) AS base30_impr
  FROM {REL}
  WHERE report_date >= DATE '{D1}' - INTERVAL 90 DAY AND report_date < DATE '{D1}'
  GROUP BY content_hash_id HAVING SUM(gsc_impressions) > 0),
future AS (
  SELECT content_hash_id, SUM(gsc_impressions) AS future_impr FROM {REL}
  WHERE report_date >= DATE '{D1}' AND report_date < DATE '{D1}' + INTERVAL 30 DAY
  GROUP BY content_hash_id)
SELECT p.*, COALESCE(f.future_impr, 0) AS future_impr
FROM prior p LEFT JOIN future f USING (content_hash_id)
ORDER BY p.content_hash_id"""

df = con.sql(q).df()
dim = con.sql(f"""SELECT content_hash_id, content_created_date,
                         content_type, main_intent, competition_level,
                         search_volume, cpc, competition, backlinks,
                         keyword_char_count, keyword_token_count,
                         url_char_count, category_count,
                         keyword_created_date, last_optimized_date,
                         optimization_eligible_date
                  FROM read_parquet('{dim_file}')""").df()
df = df.merge(dim, on="content_hash_id", how="left")
for c in ["older60_impr", "recent30_impr", "base30_impr"]:
    df[c] = df[c].fillna(0)

DECISION = pd.Timestamp(D1)
df["baseline_daily"] = df["older60_impr"] / 60
df["recent_daily"] = df["recent30_impr"] / 30
df["future_daily"] = df["future_impr"] / 30
df["target"] = np.arcsinh(df["future_daily"]) - np.arcsinh(df["baseline_daily"])
df["declined"] = df["target"] < 0
df["avg_position"] = df["sum_position_90d"] / df["impr_90d"].replace(0, np.nan) + 1
df["ctr"] = df["clicks_90d"] / df["impr_90d"].replace(0, np.nan) * 100
df["log_impr_90d"] = np.log1p(df["impr_90d"])
df["engaged_rate"] = df["ga4_engaged_90d"] / df["ga4_sessions_90d"].replace(0, np.nan)
df["pages_per_session"] = df["ga4_pageviews_90d"] / df["ga4_sessions_90d"].replace(0, np.nan)
df["scroll_per_session"] = df["scroll_events_90d"] / df["ga4_sessions_90d"].replace(0, np.nan)
df["log_ga4_sessions"] = np.log1p(df["ga4_sessions_90d"].fillna(0))
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["days_with_impressions"] = df["days_with_impressions"].fillna(0)
df["content_age_days"] = (DECISION - pd.to_datetime(df["content_created_date"])).dt.days
df["slip"] = np.where(df["baseline_daily"] > 0,
                      (df["baseline_daily"] - df["recent_daily"]) / df["baseline_daily"], np.nan)
df["peak_ratio"] = np.where(df["impr_90d"] > 0,
                            df["recent_daily"] / (df["impr_90d"] / 90), np.nan)
df["prior_trend"] = np.where(df["base30_impr"] > 0,
                             (df["recent30_impr"] - df["base30_impr"]) / df["base30_impr"], np.nan)

# --- the frozen ML-07 gate, verbatim
MIN_AGE_DAYS = 180
client_median_impr = df.groupby("client_hash_id")["impr_90d"].transform("median")
gate = ((df["baseline_daily"] > 0)
        & (df["content_age_days"] >= MIN_AGE_DAYS)
        & (df["impr_90d"] >= client_median_impr)
        & (df["slip"].fillna(0) <= 0.5))
df["in_gate"] = gate
pool = df[gate].copy()

print(f"cohort {len(df):,} pages | {df['client_hash_id'].nunique()} clients")
print(f"gated pool {len(pool):,} pages | {pool['client_hash_id'].nunique()} clients")
print(f"cohort decline rate {df['declined'].mean():.4f} | pool {pool['declined'].mean():.4f}")
print()
print("ML-07 reported: cohort 202,073 / 53 clients, pool 46,061, "
      "cohort rate 0.4228, pool rate 0.5128")

from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from scipy.stats import spearmanr

SAFE = ["content_age_days", "impr_90d", "avg_position"]
FULL = SAFE + ["prior_trend", "peak_ratio"]
K = 100
ev = pool.dropna(subset=FULL).copy()


def rank_agreement(pred, actual, ascending):
    rho = spearmanr(pred, actual).statistic
    return rho if ascending else -rho


print()
print(f"evaluation pool {len(ev):,} pages | {ev['client_hash_id'].nunique()} clients "
      f"| decline rate {ev['declined'].mean():.4f}")
print("ML-08 reported: 45,095 pages, 30 clients, 0.5131")

LEAN = [f for f in FULL if f != "content_age_days"]


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\user\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


C:\Users\user\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


python 3.13.14 | pandas 3.0.1 | numpy 2.4.4 | scikit-learn 1.9.0


cohort 202,073 pages | 53 clients
gated pool 46,061 pages | 30 clients
cohort decline rate 0.4228 | pool 0.5128

ML-07 reported: cohort 202,073 / 53 clients, pool 46,061, cohort rate 0.4228, pool rate 0.5128



evaluation pool 45,095 pages | 30 clients | decline rate 0.5131
ML-08 reported: 45,095 pages, 30 clients, 0.5131


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2a. Does one model serve every client, or is the population two populations?

ML-08's error analysis found per-client `P@100` spanning **0.458 to 0.970** on a single held-out
split, with the worst client taking 72 picks — large enough that it is not a sample-size artefact.
That is a validation question, and it is the one ML-09 should open with.

**What ML-08 already ruled out.** It tested whether *data-rich clients gain from features the others
lack*: inside the 19 of 30 clients holding keyword data, adding `search_volume`, `cpc` and
`competition` cost **−0.0072** spearman. So segmentation-by-available-columns does not help.

**What it never asked.** Whether clients differ in the *relationship itself* — the same features
predicting differently from one client to the next. If they do, a shared model is averaging over
populations that should be modelled apart, and per-client variance is structural rather than noise.

**Find the segmentation variable, do not assume it.** Score every client across all ten splits, then
ask what client-level property predicts where the model fails. If nothing does, the variance is noise
and one model is correct. If something does, that property is the segmentation candidate — and only
then is a segmented model worth building.

In [3]:
# Every client appears in the test set of roughly two splits out of ten.
# Pool those appearances so each client gets a performance record.
per_client_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te].copy()
    sc_ = StandardScaler().fit(train[FULL])
    m = Ridge(alpha=1.0).fit(sc_.transform(train[FULL]), train["target"])
    test["pred"] = m.predict(sc_.transform(test[FULL]))
    for cid, g in test.groupby("client_hash_id"):
        k = min(K, len(g))
        top = g.nsmallest(k, "pred")
        rnd = g.sample(k, random_state=seed)
        per_client_rows.append({
            "client_hash_id": cid, "seed": seed, "pages": len(g),
            "picks": k,
            "p_at_k": top["declined"].mean(),
            "p_random": rnd["declined"].mean(),
            "base_rate": g["declined"].mean(),
            "spearman": rank_agreement(g["pred"].values, g["target"].values, True)
                        if len(g) > 20 else np.nan,
        })

pc = pd.DataFrame(per_client_rows)
prof = (pc.groupby("client_hash_id")
        .agg(appearances=("seed", "nunique"), pages=("pages", "mean"),
             picks=("picks", "mean"), p_at_k=("p_at_k", "mean"),
             p_random=("p_random", "mean"), base_rate=("base_rate", "mean"),
             spearman=("spearman", "mean")))
prof["lift"] = prof["p_at_k"] - prof["p_random"]
prof = prof.sort_values("lift")

print(f"{len(prof)} clients scored, {prof['appearances'].sum()} client-appearances "
      f"across 10 splits")
print()
print(prof.round(4).to_string())

27 clients scored, 60 client-appearances across 10 splits

                         appearances   pages  picks  p_at_k  p_random  base_rate  spearman    lift
client_hash_id                                                                                    
client_08d2847f24cf89c1            2    56.0   56.0  0.5714    0.5714     0.5714    0.4563  0.0000
client_0e1acc6cd57b0eba            2     6.0    6.0  0.5000    0.5000     0.5000       NaN  0.0000
client_599043c0ff13edea            3     5.0    5.0  0.6000    0.6000     0.6000       NaN  0.0000
client_8dbf3abdf07569e0            3    13.0   13.0  0.3846    0.3846     0.3846       NaN  0.0000
client_764ae36a94e30a25            3     1.0    1.0  0.0000    0.0000     0.0000       NaN  0.0000
client_795153d5b7850ccf            3    75.0   75.0  0.7733    0.7733     0.7733    0.3438  0.0000
client_59256b0571e0c970            3     2.0    2.0  0.5000    0.5000     0.5000       NaN  0.0000
client_a1203ffecad62470            3     1.0    1.

In [4]:
# Three candidate explanations for the per-client spread, tested in order of
# how boring they are. A structural cause has to be ruled out before
# "clients differ" is allowed to stand.
prof2 = prof.copy()
prof2["rankable"] = prof2["pages"] > K
prof2["br_distance"] = (prof2["base_rate"] - 0.5).abs()

zero = prof2[prof2["lift"] <= 0.0001]
print(f"clients with exactly zero lift: {len(zero)} of {len(prof2)}")
print(f"  of those, how many have pages <= K={K}: {(zero['pages'] <= K).sum()}")
print(f"  their page counts: {sorted(zero['pages'].astype(int))}")
print()
print("When a client has fewer pages than the queue has slots, the queue takes")
print("every page and no ranking happens. Lift is zero by construction, not by")
print("failure -- and precision equals that client's own base rate exactly.")
print()

rank_ok = prof2[prof2["rankable"]]
print(f"among the {len(rank_ok)} clients with more pages than slots:")
print(f"  lift range {rank_ok['lift'].min():.4f} - {rank_ok['lift'].max():.4f}")
print(f"  Spearman(lift, |base_rate - 0.5|) = "
      f"{rank_ok['lift'].corr(rank_ok['br_distance'], method='spearman'):+.4f}")
print(f"  Spearman(lift, pages)             = "
      f"{rank_ok['lift'].corr(rank_ok['pages'], method='spearman'):+.4f}")
print()
print("A base rate far from 0.5 caps how much any ranking can add: at 0.91 a")
print("random queue already scores 0.91 and only 0.09 of headroom exists.")
print()

# The capacity-free, base-rate-free measure: does the model order pages
# correctly INSIDE each client, regardless of how many slots the queue has?
sp = prof2["spearman"].dropna()
print(f"per-client rank agreement, the measure neither capacity nor base rate distorts:")
print(f"  clients with enough pages to compute it: {len(sp)} of {len(prof2)}")
print(f"  range {sp.min():.4f} - {sp.max():.4f} | median {sp.median():.4f}")
print(f"  clients with NEGATIVE rank agreement: {(sp < 0).sum()}")

clients with exactly zero lift: 12 of 27
  of those, how many have pages <= K=100: 12
  their page counts: [1, 1, 2, 5, 6, 13, 22, 56, 72, 75, 78, 100]

When a client has fewer pages than the queue has slots, the queue takes
every page and no ranking happens. Lift is zero by construction, not by
failure -- and precision equals that client's own base rate exactly.

among the 15 clients with more pages than slots:
  lift range 0.0100 - 0.5700
  Spearman(lift, |base_rate - 0.5|) = -0.4714
  Spearman(lift, pages)             = +0.5321

A base rate far from 0.5 caps how much any ranking can add: at 0.91 a
random queue already scores 0.91 and only 0.09 of headroom exists.

per-client rank agreement, the measure neither capacity nor base rate distorts:
  clients with enough pages to compute it: 21 of 27
  range 0.3438 - 0.7852 | median 0.4814
  clients with NEGATIVE rank agreement: 0


**Verdict: the per-client spread is capacity and base rates. The model works for every client, and
ML-08's "worst client" finding is withdrawn.**

**Twelve of 27 clients have exactly zero lift, and all twelve have fewer pages than the queue has
slots** — their page counts are 1, 1, 2, 5, 6, 13, 22, 56, 72, 75, 78 and 100. With `pages <= K = 100` the queue takes the whole client, no ranking occurs, and precision
equals that client's own base rate by arithmetic.

**That includes ML-08's worst client.** `client_d211cb07b9059bab` scored **0.458** — and its base rate
is **0.4583**. It has **72 pages** against 100 slots. ML-08 reported it as a model weakness and argued
it was not a sample-size artefact because 72 picks is a lot. **That reasoning was wrong.** It is not a
sample-size artefact; it is a *capacity* artefact. The model never ranked anything for that client,
because there was nothing to rank.

**Among the 15 clients that can actually be ranked, lift tracks the base rate rather than client
character** — `Spearman(lift, |base_rate - 0.5|)` is **-0.4714**, and `Spearman(lift, pages)` is
**+0.5321**. Neither alone explains it; together with the twelve structural zeros they leave little
for client character to account for. A client whose pages decline at 0.9138 leaves only 0.09 of headroom above random, and
scores 0.0833. A client at 0.1662 leaves plenty, and scores 0.5567.

**The measure that survives both distortions is per-client rank agreement.** It does not care how many
slots the queue has or where the base rate sits — it asks only whether the model orders that client's
pages correctly. **It is positive for all 21 clients where it can be computed** — range 0.3438 to 0.7852, median
**0.4814**, and **zero** clients negative.

**So the segmented model is not justified, and for a better reason than ML-08 gave.** ML-08 showed
that data-rich clients gain nothing from the extra columns they have. This shows something stronger:
**clients do not differ in the relationship at all.** One model is correct because there is one
relationship, not because segmentation was impractical.

**What this costs, and it is worth stating in the recommendation.** Twelve of 27 clients receive a
queue that is simply their whole eligible page list. For them the product is the *gate*, not the
model — which matches ML-07's finding that the gate carries information and the ordering did not.
Selling those clients a "ranked queue" would misdescribe what they get.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Leakage audit on the final five features

The same hunt as ML-05, on what actually ships: `peak_ratio`, `prior_trend`, `content_age_days`,
`impr_90d`, `avg_position`.

**The order matters.** A positive control runs *first*. If deliberately injecting the answer does not
send the score toward 1.0, the harness cannot detect leakage and every result below it is
uninterpretable — including the clean ones. A leakage audit that has never seen leakage has proved
nothing.

Then the attack checklist: timeline, ablation on the dominant feature, population selection, and
product flags.

In [6]:
# ---------------------------------------------------------------- 1. positive control
# Inject the answer. If the harness works, these must approach 1.0.
LEAKS = {
    "target itself": "target",
    "future_daily (the label's numerator)": "future_daily",
    "declined (the evaluation label)": "declined",
}
ctrl = []
for seed in range(5):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]
    for label, col in [("none (the shipped model)", None)] + list(LEAKS.items()):
        feats = FULL if col is None else FULL + [col]
        sc_ = StandardScaler().fit(train[feats])
        m = Ridge(alpha=1.0).fit(sc_.transform(train[feats]), train["target"])
        pred = m.predict(sc_.transform(test[feats]))
        ctrl.append({"injected": label,
                     "spearman": rank_agreement(pred, test["target"].values, True),
                     "auc": roc_auc_score(test["declined"], -pred)})

cc = pd.DataFrame(ctrl).groupby("injected").mean().round(4)
print("POSITIVE CONTROL -- inject the answer, five grouped splits")
print(cc.to_string())
clean = cc.loc["none (the shipped model)", "spearman"]
full_leak = cc.loc["target itself", "spearman"]
label_leak_auc = cc.loc["declined (the evaluation label)", "auc"]
print()
print(f"harness detects a full leak: {full_leak > 0.99} (target injected -> {full_leak:.4f})")
print(f"harness detects a label leak: {label_leak_auc > 0.99} "
      f"(declined injected -> AUC {label_leak_auc:.4f})")
print(f"shipped model sits at {clean:.4f}, far below either")
print()
print("A partial leak leaks partially. future_daily reaches only "
      f"{cc.loc['future_daily (the label\'s numerator)', 'spearman']:.4f} because the target is")
print("asinh(future_daily) - asinh(baseline_daily): one term of two is not the answer.")
print("An earlier version of this cell demanded EVERY injection exceed 0.9 and")
print("printed 'harness works: False'. That criterion was wrong, not the harness.")
print()

# ---------------------------------------------------------------- 2. timeline
print("TIMELINE -- every feature must be knowable at the decision point")
DEC = pd.Timestamp(D1)
created = pd.to_datetime(df["content_created_date"], errors="coerce")
print(f"  content_created_date after {D1}: "
      f"{int((created > DEC).sum()):,} of {int(created.notna().sum()):,}")
print(f"  latest created_date: {created.max().date()}")
print("  impr_90d, avg_position, peak_ratio, prior_trend: all summed over")
print(f"    [{D1} - 90d, {D1}) by the query's WHERE clause -- no future rows reachable")
print(f"  target window: [{D1}, {D1} + 30d) -- strictly after, no overlap")
print()

# ---------------------------------------------------------------- 3. ablation
print("ABLATION -- drop each feature in turn, ten grouped splits")
abl = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]
    for drop in [None] + FULL:
        feats = FULL if drop is None else [f for f in FULL if f != drop]
        sc_ = StandardScaler().fit(train[feats])
        m = Ridge(alpha=1.0).fit(sc_.transform(train[feats]), train["target"])
        pred = m.predict(sc_.transform(test[feats]))
        abl.append({"dropped": drop or "nothing",
                    "spearman": rank_agreement(pred, test["target"].values, True)})

ab = pd.DataFrame(abl).groupby("dropped")["spearman"].mean().round(4).sort_values()
print(ab.to_string())
print()

# ---------------------------------------------------------------- 4. population + flags
print("POPULATION SELECTION -- does the gate read the outcome window?")
gate_inputs = {"baseline_daily": "days 30-90 before the decision point",
               "content_age_days": "created_date, immutable, vs the decision date",
               "impr_90d": "the 90 days before the decision point",
               "slip": "recent 30d vs days 30-90, both before"}
for k, v in gate_inputs.items():
    print(f"  {k:18s} {v}")
print("  none reads the label window. The cohort keeps pages that reach zero,")
print("  so survival in the outcome month is NOT a selection criterion.")
print()
print("PRODUCT FLAGS -- FlyRank's own scores must never be inputs")
flagish = [c for c in ev.columns
           if any(t in c.lower() for t in ("flag", "risk", "health", "score", "priority",
                                           "optimiz", "eligible"))]
print(f"  flag-like columns present in the frame: {flagish}")
print(f"  any of them in FULL: {[f for f in FULL if f in flagish]}")

# ---------------------------------------------------------------- 5. the four-feature model
# The ablation says dropping content_age_days IMPROVES the model, which the
# permutation importance (-0.0645) predicted. Check it on the full metric
# suite before recommending a change to what ships.
LEAN = [f for f in FULL if f != "content_age_days"]
lean_rows = []
for seed in range(10):
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                  .split(ev, groups=ev["client_hash_id"]))
    train, test = ev.iloc[tr], ev.iloc[te]
    for label, feats in [("FULL (5)", FULL), ("LEAN (4, no age)", LEAN)]:
        sc_ = StandardScaler().fit(train[feats])
        m = Ridge(alpha=1.0).fit(sc_.transform(train[feats]), train["target"])
        pred = m.predict(sc_.transform(test[feats]))
        hits = picks = 0
        for _, g in test.assign(s=pred).groupby("client_hash_id"):
            top = g.nsmallest(min(K, len(g)), "s")
            hits += int(top["declined"].sum()); picks += len(top)
        lean_rows.append({"features": label,
                          "spearman": rank_agreement(pred, test["target"].values, True),
                          "p_at_100": hits / picks,
                          "auc": roc_auc_score(test["declined"], -pred)})

ln = pd.DataFrame(lean_rows).groupby("features").mean().round(4)
print()
print("DROPPING content_age_days -- full metric suite, ten grouped splits")
print(ln.to_string())
print()
for m_ in ["spearman", "p_at_100", "auc"]:
    print(f"  {m_:10s} {ln.loc['LEAN (4, no age)', m_] - ln.loc['FULL (5)', m_]:+.4f}")

POSITIVE CONTROL -- inject the answer, five grouped splits
                                      spearman     auc
injected                                              
declined (the evaluation label)         0.8039  0.9993
future_daily (the label's numerator)    0.6596  0.8267
none (the shipped model)                0.5498  0.7625
target itself                           0.9999  1.0000

harness detects a full leak: True (target injected -> 0.9999)
harness detects a label leak: True (declined injected -> AUC 0.9993)
shipped model sits at 0.5498, far below either

A partial leak leaks partially. future_daily reaches only 0.6596 because the target is
asinh(future_daily) - asinh(baseline_daily): one term of two is not the answer.
An earlier version of this cell demanded EVERY injection exceed 0.9 and
printed 'harness works: False'. That criterion was wrong, not the harness.

TIMELINE -- every feature must be knowable at the decision point
  content_created_date after 2026-03-31: 0 of 202,0

dropped
peak_ratio          0.1192
impr_90d            0.5214
nothing             0.5302
prior_trend         0.5305
avg_position        0.5306
content_age_days    0.5546

POPULATION SELECTION -- does the gate read the outcome window?
  baseline_daily     days 30-90 before the decision point
  content_age_days   created_date, immutable, vs the decision date
  impr_90d           the 90 days before the decision point
  slip               recent 30d vs days 30-90, both before
  none reads the label window. The cohort keeps pages that reach zero,
  so survival in the outcome month is NOT a selection criterion.

PRODUCT FLAGS -- FlyRank's own scores must never be inputs
  flag-like columns present in the frame: ['last_optimized_date', 'optimization_eligible_date']
  any of them in FULL: []



DROPPING content_age_days -- full metric suite, ten grouped splits
                  spearman  p_at_100     auc
features                                    
FULL (5)            0.5302    0.7563  0.7584
LEAN (4, no age)    0.5546    0.7612  0.7693

  spearman   +0.0244
  p_at_100   +0.0049
  auc        +0.0109


**Verdict: no leakage found, and the audit removes a feature. The model ships with four, not five.**

**The positive control passes, which is what makes everything below it readable.**

| injected | spearman | AUC |
|---|---|---|
| **target itself** | **0.9999** | **1.0000** |
| `declined` (the evaluation label) | 0.8039 | **0.9993** |
| `future_daily` (one of the target's two terms) | 0.6596 | 0.8267 |
| **none — the shipped model** | **0.5498** | 0.7625 |

Injecting the answer produces near-perfection, so the harness can see leakage. The shipped model sits
at **0.5498** against a measured leak level of **0.9999** — not a near miss.

**A partial leak leaks partially, and that is the most useful thing this control taught.**
`future_daily` reaches only 0.6596, because `target = asinh(future_daily) − asinh(baseline_daily)` and
one term of two is not the answer. **A feature overlapping half the target's construction would not
announce itself loudly** — which is exactly the failure mode that cost this project three findings
before the null simulation caught it. Injection tests find blatant leaks; they do not find subtle ones.

> ⚠️ **This cell first printed `harness works: False`.** The criterion demanded *every* injection exceed
> 0.9, and `future_daily` does not. The criterion was wrong, not the harness. Recorded because a
> failing self-test that is itself broken is the worst possible thing to skim past.

**`peak_ratio` dominance is not leakage.** Dropping it collapses spearman from 0.5302 to **0.1192**,
which satisfies half the skill's rule — one feature towering over the rest. It fails the other half:
the rule requires a *near-perfect* score, and 0.5302 against a leak level of 0.9999 is nowhere near.
The null simulation already established the same thing from the other direction, with ~81% of the gain
surviving a future carrying no information.

**Timeline, population and flags are clean.** No `content_created_date` falls after the decision point
— **0 of 202,073**, latest value exactly 2026-03-31. Every windowed feature is bounded by the query's
`WHERE` clause. The gate reads `baseline_daily`, `content_age_days`, `impr_90d` and `slip`, all
pre-decision, and the cohort keeps pages that reach zero — so survival in the outcome month is not a
selection criterion. The two flag-like columns in the frame, `last_optimized_date` and
`optimization_eligible_date`, are excluded from the feature set.

**The audit changes what ships.**

| dropped | spearman |
|---|---|
| `peak_ratio` | 0.1192 |
| `impr_90d` | 0.5214 |
| nothing | 0.5302 |
| `prior_trend` | 0.5305 |
| `avg_position` | 0.5306 |
| **`content_age_days`** | **0.5546** |

Removing `content_age_days` improves every metric:

| | FULL (5) | LEAN (4) | change |
|---|---|---|---|
| spearman | 0.5302 | **0.5546** | **+0.0244** |
| `P@100` | 0.7563 | **0.7612** | +0.0049 |
| AUC | 0.7584 | **0.7693** | +0.0109 |

Direct ablation confirming what permutation importance predicted at **−0.0645**. Two independent
methods agree the feature is not merely useless but harmful — it correlates −0.1981 with the target
and is collinear enough with `peak_ratio` to split its coefficient.

**ML-08 audited twenty candidates down to five. Validation removes a sixth.** That is the process
working rather than failing: ML-08's comparison table never dropped features one at a time, because
it was comparing model classes rather than feature sets. The ablation is cheap and should have run
there.

### The one segment ML-08 never tested: GA4

ML-08 ran the segment protocol on three groups — keyword, links, structural. **GA4 was excluded by
argument rather than by test**, on the grounds that ML-06 Test 9 had already measured per-client
engagement correlations at a median of **+0.037** and **+0.015** and found 0 of 13 and 2 of 24 clients
above \|ρ\| = 0.2.

**That argument does not hold, for the reason this project has demonstrated more than any other:
correlation barely predicts contribution.** `prior_trend` correlates **+0.5438** with the target and
contributes **0.0038**; `url_char_count` correlates **−0.2225** and contributes **−0.0185**. Inferring
"will not help a model" from "does not correlate" is the inference ML-08 disproved five times over.

Test 9 also measured on ML-06's cohort, not the gated evaluation pool, and never fit a model. Three
differences from the protocol the other groups received.

**Same test as the others, on the four-feature model that now ships.**

In [7]:
GA4 = ["engaged_rate", "pages_per_session", "scroll_per_session", "log_ga4_sessions"]
GA4 = [c for c in GA4 if c in ev.columns]
assert GA4, "GA4 features not built -- an empty result here is a bug, not a finding"

have = ev[GA4].notna().all(axis=1) & (ev["ga4_sessions_90d"].fillna(0) > 0)
per = ev.assign(h=have).groupby("client_hash_id")["h"].mean()
rich = per[per > 0.95].index
sub = ev[ev["client_hash_id"].isin(rich) & have].copy()

print(f"GA4 features: {GA4}")
print(f"clients with >95% GA4 coverage: {len(rich)} of {ev['client_hash_id'].nunique()}")
print(f"pages in the GA4-rich segment: {len(sub):,} | decline rate {sub['declined'].mean():.4f}")
print()
for f in GA4:
    ok = sub[f].notna() & np.isfinite(sub[f])
    print(f"  Spearman({f:20s}, target) = "
          f"{sub.loc[ok, f].corr(sub.loc[ok, 'target'], method='spearman'):+.4f}")

if sub["client_hash_id"].nunique() < 8:
    print(f"\nonly {sub['client_hash_id'].nunique()} clients -- too few to hold out")
else:
    rows_g = []
    for seed in range(10):
        tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
                      .split(sub, groups=sub["client_hash_id"]))
        train, test = sub.iloc[tr], sub.iloc[te]
        for label, feats in [("LEAN (shared model)", LEAN), ("LEAN + GA4", LEAN + GA4)]:
            t2, e2 = train.dropna(subset=feats), test.dropna(subset=feats)
            if e2.empty:
                continue
            sc_ = StandardScaler().fit(t2[feats])
            m = Ridge(alpha=1.0).fit(sc_.transform(t2[feats]), t2["target"])
            pred = m.predict(sc_.transform(e2[feats]))
            hits = picks = 0
            for _, g in e2.assign(s=pred).groupby("client_hash_id"):
                top = g.nsmallest(min(K, len(g)), "s")
                hits += int(top["declined"].sum()); picks += len(top)
            rows_g.append({"features": label,
                           "spearman": rank_agreement(pred, e2["target"].values, True),
                           "p_at_100": hits / picks,
                           "auc": roc_auc_score(e2["declined"], -pred)})

    gg = pd.DataFrame(rows_g).groupby("features").mean().round(4)
    print()
    print("inside the GA4-rich clients only, ten grouped splits:")
    print(gg.to_string())
    print()
    print("what GA4 buys, inside the clients that have it:")
    for m_ in ["spearman", "p_at_100", "auc"]:
        print(f"  {m_:10s} {gg.loc['LEAN + GA4', m_] - gg.loc['LEAN (shared model)', m_]:+.4f}")

GA4 features: ['engaged_rate', 'pages_per_session', 'scroll_per_session', 'log_ga4_sessions']
clients with >95% GA4 coverage: 6 of 30
pages in the GA4-rich segment: 8,756 | decline rate 0.4076

  Spearman(engaged_rate        , target) = -0.0062
  Spearman(pages_per_session   , target) = -0.1841
  Spearman(scroll_per_session  , target) = -0.0886
  Spearman(log_ga4_sessions    , target) = -0.0635

only 6 clients -- too few to hold out


**Verdict: the GA4 segment cannot be tested at this scale — and the reason corrects ML-07.**

**Only 6 of 30 clients have >95% GA4 coverage.** ML-07's availability check reported GA4 present for
29 of 36 clients with just 6 at zero, and this notebook repeated that as "24 clients could get a
richer model". Both were wrong.

**That check measured `notna()`.** `SUM(ga4_sessions)` returns **0** for a page with no GA4 traffic,
not null — so "available" meant "the column exists", which is true of nearly every row. Requiring
actual sessions, which any engagement *rate* needs as a denominator, collapses coverage from 29
clients to **6**.

**So the segment fails the same bar backlinks failed at 4 clients.** Holding out 20% of six clients
leaves one, and training on five. Any number from that would be noise wearing a result's clothes.

**What the correlations show inside the 6 clients that do have it**, on 8,756 pages:

| feature | Spearman vs target |
|---|---|
| `pages_per_session` | **−0.1841** |
| `scroll_per_session` | −0.0886 |
| `log_ga4_sessions` | −0.0635 |
| `engaged_rate` | −0.0062 |

`pages_per_session` at −0.1841 is comparable to `content_age_days` cohort-wide, so this is not
obviously nothing — and it is exactly the kind of correlation ML-08 showed can contribute zero. It
cannot be resolved either way here.

**The honest position.** GA4 is excluded because **too few clients carry usable data to validate it**,
not because it was shown not to help. ML-06 Test 9 measured correlation on a different cohort and
never fit a model; this notebook's earlier claim that Test 9 settled the question was wrong. The
question is open and will stay open until more clients have GA4 instrumented.

**A note for the data contract.** Distinguishing "column present" from "data present" matters
wherever a rate is computed. A denominator of zero is not missing data — it is a different kind of
absent, and an availability audit that counts nulls will not see it.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.